
### Notebook : 02_shared_state_and_helpers


##### 1 : Purpose

This notebook defines the shared workflow state and reusable state-management functions.

It contains:

- MultiAgentState
- create_initial_state()
- get_agent_task()
- record_agent_execution()
- record_agent_error()

It does not execute any specialist tools or agents.


##### 2: Technologies Used

- Python
- TypedDict
- Pydantic models from 01_shared_schemas
- Type hints
- Reusable helper functions
- Databricks %run


##### 3 : Input

This notebook supports the following inputs:

- Customer request
- Coordinator execution-plan tasks
- Agent execution status and messages
- Workflow error details


##### 4 : Output

This notebook produces:
- Initialized MultiAgentState
- Assigned AgentTask or None
- Validated ExecutionRecord entries
- Standardized workflow-error entries

##### 5: Architecture

``` text

Customer request
       │
       ▼
create_initial_state()
       │
       ▼
MultiAgentState
       │
       ├── user_request
       ├── coordinator_result
       ├── agent_results
       ├── execution_history
       ├── final_response
       └── errors
               │
               ▼
      Reusable helper functions

```


##### 6 : Load shared schemas

In [0]:
%run ./01_shared_models_code_only


##### 7 : Additional imports

In [0]:
from typing import Any, Dict, List, Optional, TypedDict

##### 8 : Shared-state definition

In [0]:
class MultiAgentState(TypedDict):
    """
    Shared state passed between agents in the multi-agent workflow.
    """

    user_request: str

    coordinator_result: Optional[CoordinatorResult]

    agent_results: Dict[AgentName, BaseAgentResult]

    execution_history: List[ExecutionRecord]

    final_response: Optional[str]

    errors: List[Dict[str, Any]]

- user_request - Original customer request.

- coordinator_result  -  Validated execution plan produced by the Coordinator Agent.

- agent_results  -  Validated outputs returned by the specialist agents.

- execution_history - Ordered history of successful, failed, or skipped executions.

- final_response - Final customer-facing answer.

- errors - Workflow and agent errors collected during execution.


##### 9 : Initial-state function

In [0]:
def create_initial_state(
    user_request: str,
) -> MultiAgentState:
    """
    Create the initial shared state for one customer request.

    Parameters
    ----------
    user_request:
        Original request submitted by the customer.

    Returns
    -------
    MultiAgentState
        New shared state containing empty workflow results.

    Raises
    ------
    ValueError
        If the request is empty or contains only whitespace.
    """

    cleaned_request = user_request.strip()

    if not cleaned_request:
        raise ValueError(
            "user_request must contain at least one "
            "non-whitespace character."
        )

    return {
        "user_request": cleaned_request,
        "coordinator_result": None,
        "agent_results": {},
        "execution_history": [],
        "final_response": None,
        "errors": [],
    }


##### 10 : Helper: Find an assigned task

In [0]:
def get_agent_task(
    tasks: List[AgentTask],
    agent_name: AgentName,
) -> Optional[AgentTask]:
    """
    Return the task assigned to a specified agent.

    Parameters
    ----------
    tasks:
        Execution-plan tasks created by the Coordinator Agent.

    agent_name:
        Specialist agent looking for its assigned task.

    Returns
    -------
    Optional[AgentTask]
        Assigned task when found; otherwise None.
    """

    for task in tasks:
        if task.agent_name == agent_name:
            return task

    return None


##### 11 : Helper: Record Agent Execution

In [0]:
def record_agent_execution(
    state: MultiAgentState,
    agent_name: AgentName,
    status: AgentStatus,
    message: str,
) -> None:
    """
    Add one validated execution event to shared state.

    This function modifies the shared-state dictionary directly.
    Therefore, it does not return the state.

    Parameters
    ----------
    state:
        Current multi-agent workflow state.

    agent_name:
        Agent associated with the execution event.

    status:
        Execution status of the agent.

    message:
        Short description of the execution event.
    """

    execution_record = ExecutionRecord(
        agent_name=agent_name,
        status=status,
        message=message,
    )

    state["execution_history"].append(
        execution_record
    )


##### 12 : Helper: Record Agent or Workflow Error

In [0]:

def record_agent_error(
    state: MultiAgentState,
    agent_name: AgentName,
    error_code: str,
    error_message: str,
) -> None:
    """
    Add one validated agent or workflow error to shared state.

    This function modifies the shared-state dictionary directly.
    Therefore, it does not return the state.

    Parameters
    ----------
    state:
        Current multi-agent workflow state.

    agent_name:
        Agent associated with the error.

    error_code:
        Standardized machine-readable error code.

    error_message:
        Human-readable description of the error.
    """

    error_record = AgentErrorRecord(
        agent_name=agent_name,
        error_code=error_code,
        error_message=error_message,
    )

    state["errors"].append(
        error_record
    )


##### 13 : Independent Test Function

In [0]:
def test_shared_state_and_helpers() -> None:
    """
    Run independent validation tests for shared-state
    initialization and reusable helper functions.
    """

    test_state = create_initial_state(
        "Video streaming keeps buffering."
    )

    test_tasks = [
        AgentTask(
            task_id="task_1",
            agent_name="prediction_agent",
            task_description=(
                "Predict the support-ticket category."
            ),
        ),
        AgentTask(
            task_id="task_2",
            agent_name="final_response_agent",
            task_description=(
                "Generate the final customer-facing response."
            ),
            depends_on=["prediction_agent"],
        ),
    ]

    assigned_task = get_agent_task(
        tasks=test_tasks,
        agent_name="prediction_agent",
    )

    missing_task = get_agent_task(
        tasks=test_tasks,
        agent_name="retention_agent",
    )

    record_agent_execution(
        state=test_state,
        agent_name="prediction_agent",
        status="success",
        message="Prediction task completed successfully.",
    )

    record_agent_error(
        state=test_state,
        agent_name="prediction_agent",
        error_code="TEST_ERROR",
        error_message=(
            "Example error used to validate error recording."
        ),
    )

    assert (
        test_state["user_request"]
        == "Video streaming keeps buffering."
    )

    assert (
        test_state["coordinator_result"]
        is None
    )

    assert (
        test_state["agent_results"]
        == {}
    )

    assert assigned_task is not None

    assert (
        assigned_task.agent_name
        == "prediction_agent"
    )

    assert missing_task is None

    assert len(
        test_state["execution_history"]
    ) == 1

    assert (
        test_state["execution_history"][0].agent_name
        == "prediction_agent"
    )

    assert (
        test_state["execution_history"][0].status
        == "success"
    )

    assert len(
        test_state["errors"]
    ) == 1

    assert (
        test_state["errors"][0].error_code
        == "TEST_ERROR"
    )

    try:
        create_initial_state("   ")

    except ValueError:
        pass

    else:
        raise AssertionError(
            "An empty request should raise ValueError."
        )

    print(
        "All shared-state and helper tests passed."
    )

In [0]:
# Optional Inspection Example

example_state = create_initial_state(
    "How many Billing tickets are there?"
)

record_agent_execution(
    state=example_state,
    agent_name="sql_agent",
    status="success",
    message="SQL analytics task completed successfully.",
)

record_agent_error(
    state=example_state,
    agent_name="sql_agent",
    error_code="EXAMPLE_ERROR",
    error_message="Example error record.",
)

print("Execution History:")

for record in example_state["execution_history"]:
    print(record.model_dump())

print("\nErrors:")

for error in example_state["errors"]:
    print(error.model_dump())


##### 14 : Key Learnings

- TypedDict documents the complete structure of the shared workflow state.

- Shared state allows agents to exchange structured information without relying on unstructured conversation history.

- create_initial_state() ensures that every customer request begins with an isolated and valid state object.

- Reusable helper functions prevent duplicated state-management logic across agent notebooks.

- get_agent_task() separates execution-plan lookup from specialist-agent logic.

- The current design assumes that each specialist agent receives at most one task.

- record_agent_execution() creates consistent and validated workflow-history records.

- record_agent_error() creates consistent and validated error records.

- Helper functions can update mutable state directly and therefore do not need to return it.

- Structured agent results remain separate from the final customer-facing response.

- Consistent status, history, and error records make the workflow easier to test and debug.

##### 15 : Conclusion

- In this notebook, we created the shared state used by the multi-agent workflow and implemented reusable functions for initializing state, locating assigned tasks, recording execution history, and recording agent or workflow errors.

- Centralizing this behavior prevents the Coordinator, SQL, Prediction, Vector Search, Retention, and Final Response notebooks from duplicating state-management logic. Each downstream agent notebook can now focus only on its unique responsibility while exchanging information through the same structured workflow state.


##### 16 : Next Notebook

03_coordinator_agent

The Coordinator Agent will:

- Read the customer request.
- Identify the request type.
- Select the required specialist agents.
- Create an ordered execution plan.
- Define task dependencies.
- Return a validated CoordinatorResult.
- Store the Coordinator result in shared state.